### 1. Reading dataset

Let's first have a look at the dataset and understand the size, attribute names etc.

In [1]:
import pandas as pd
import io
# from google.colab import files
# uploaded = files.upload()

In [2]:
# cars = pd.read_csv(io.BytesIO(uploaded['CarPrice_Assignment_One_Var.csv'])) # for colab
cars = pd.read_csv(r"CarPrice_Assignment_One_Var.csv")

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
pd.options.mode.chained_assignment = None  # default='warn' (to supress warning in outlier treatment capping)
np.seterr(divide='ignore') #Ignore runtime warning in VIF (for division of zero)
pd.options.display.max_rows = 4000  #(For avoiding results geting truncated)

In [4]:
# summary of the dataset: 205 rows, 3 columns, no null values
print(cars.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   car_ID      205 non-null    int64  
 1   horsepower  205 non-null    int64  
 2   price       205 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 4.9 KB
None


In [5]:
# Lets look at the first 5 rows of he dataset
cars.head()

,car_ID,horsepower,price
0,1,111,13495.0
1,2,111,16500.0
2,3,154,16500.0
3,4,102,13950.0
4,5,115,17450.0


In [6]:
#should Car ID be integer?
cars.dtypes

car_ID          int64
horsepower      int64
price         float64
dtype: object

In [7]:
#Lets Cast Car ID as category (always convert IDs to categorical variables)
cars['car_ID'] = cars['car_ID'].astype('object')
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   car_ID      205 non-null    object 
 1   horsepower  205 non-null    int64  
 2   price       205 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 4.9+ KB


In [8]:
cars.head()

,car_ID,horsepower,price
0,1,111,13495.0
1,2,111,16500.0
2,3,154,16500.0
3,4,102,13950.0
4,5,115,17450.0


## Creating correlation i.e Pierson matrix

In [9]:
# all numeric (float and int) variables in the dataset
cars_numeric = cars.select_dtypes(include=['float64', 'int64'])
cars_numeric.head()

,horsepower,price
0,111,13495.0
1,111,16500.0
2,154,16500.0
3,102,13950.0
4,115,17450.0


In [11]:
# correlation matrix
cor = cars_numeric.corr()
cor.round(2)

,horsepower,price
horsepower,1.00,0.81
price,0.81,1.00


# 2. Data Preparation


#### Data Preparation

Let's now prepare the data and build the model.

In [12]:
cars.columns

Index(['car_ID', 'horsepower', 'price'], dtype='object')

In [18]:
# split into X and y (do not select "Car ID")
# X = cars.loc[:, ['horsepower']]     # X is input and should always be a Dataframe
X = cars[['horsepower']]   # another way to create dataframe

Y  = cars['price']   # and Y is the output i.e target and it should always be a Series

# Model Building and Evaluation

## Splitting Data in Test and Train

## Fitting Linear Model

In [19]:
# fitting a linear model
import statsmodels.api as sm

#adding a constant (As B0 cannot be calculated directly in this package so we are adding this variable which is always 1)
X = sm.add_constant(X)

# split into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.8,test_size = 0.2, random_state=100)

#performing the regression
model = sm.OLS(Y_train,X_train).fit()

#Train - Good , Test : Bad --> Overfit
#Train - Bad , Test : Bad  --> Underfit
#Train - Good , Test : Good --> Bestfit
#Train - Bad , Test : Good --> Not possible

In [ ]:
# from sklearn.linear_model import LinearRegression
# from sklearn.model_selection import train_test_split

# X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.8, test_size=0.2, random_state=100)

# # No need to add constant - sklearn does it internally
# model = LinearRegression(fit_intercept=True)  # fit_intercept=True is default
# model.fit(X_train, Y_train)

In [20]:
X_train.shape

(164, 2)

In [21]:
X_train.head()

,const,horsepower
3,1.0,102
157,1.0,70
81,1.0,88
32,1.0,60
99,1.0,97


In [23]:
X_test.shape

(41, 2)

In [40]:
X_test.head()

,const,horsepower
160,1.0,70
186,1.0,85
59,1.0,84
165,1.0,112
140,1.0,73


In [41]:
# Result of statsmodels
# OLS = Ordinary least squares
#No. Observations = Number of obeservations in train data = 164
#Df residuals = n - (k+1) = 164 - (df model +1)
#df model = In general equals to number of variables
#"const" term: The constant terms is the intercept of the regression line
# std error: std err reflects the level of accuracy of the coefficients. The lower it is, the higher is the level of accuracy
# t value: (coef/SD)
# P >|t| is your p-value. A p-value of less than 0.05 is considered to be statistically significant
# Confidence Interval represents the range in which our coefficients are likely to fall (with a likelihood of 95%)

#More details on how to read these results: https://www.geeksforgeeks.org/interpreting-the-results-of-linear-regression-using-ols-summary/#:~:text=Degree%20of%20freedom(df)%20of,sum%20of%20squares%20is%20calculated.
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.660
Model:                            OLS   Adj. R-squared:                  0.658
Method:                 Least Squares   F-statistic:                     314.9
Date:                Sat, 14 Feb 2026   Prob (F-statistic):           7.96e-40
Time:                        11:41:39   Log-Likelihood:                -1611.1
No. Observations:                 164   AIC:                             3226.
Df Residuals:                     162   BIC:                             3232.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -3283.9339    969.914     -3.386      0.0

In [ ]:
# car_price = -3283.9339 * horsepower     i.e Y = b0 + b1 * X

In [25]:
#rmse: Root Mean Square Error (Cost Function)
from statsmodels.tools.eval_measures import rmse
# fit your model which you have already done


# now generate predictions on training- question teaching in class
ypred_train = model.predict(X_train)

# calc rmse # original ans key and what students have given
rmse_train = rmse(Y_train, ypred_train)
rmse_train

4469.0283922493445

on an avg model is off 4469 for each car

In [26]:
X_train.head()

,const,horsepower
3,1.0,102
157,1.0,70
81,1.0,88
32,1.0,60
99,1.0,97


In [27]:
ypred_train.head()

3      12912.134910
157     7831.015278
81     10689.145071
32      6243.165393
99     12118.209967
dtype: float64

In [28]:
# now generate predictions
ypred_test = model.predict(X_test)

# calc rmse
rmse_test = rmse(Y_test, ypred_test)
rmse_test

5516.679395700873

# Adding duplicate variable

In [42]:
cars_2 = cars
cars_2['horsepower_2']=cars['horsepower']

In [43]:
# split into X and y (do not select "Car ID")
X2 = cars_2.loc[:, ['horsepower','horsepower_2']]
Y2  = cars_2['price']

In [31]:
# fitting a linear model
import statsmodels.api as sm

#adding a constant
X2 = sm.add_constant(X2)

# split into train and test
from sklearn.model_selection import train_test_split
X_train2, X_test2, Y_train2, Y_test2 = train_test_split(X2, Y2, train_size=0.8,test_size = 0.2, random_state=100)

#performing the regression
model2 = sm.OLS(Y_train2,X_train2).fit()

In [32]:
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.660
Model:                            OLS   Adj. R-squared:                  0.658
Method:                 Least Squares   F-statistic:                     314.9
Date:                Sat, 14 Feb 2026   Prob (F-statistic):           7.96e-40
Time:                        11:40:56   Log-Likelihood:                -1611.1
No. Observations:                 164   AIC:                             3226.
Df Residuals:                     162   BIC:                             3232.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const        -3283.9339    969.914     -3.386   

In [33]:
from statsmodels.tools.eval_measures import rmse
# fit your model which you have already done

# now generate predictions
ypred_train2 = model2.predict(X_train2)

# calc rmse
rmse_train = rmse(Y_train2, ypred_train2)
rmse_train

4469.0283922493445

In [34]:
# now generate predictions
ypred_test2 = model2.predict(X_test2)

# calc rmse
rmse_test = rmse(Y_test2, ypred_test2)
rmse_test

5516.679395700874

In [35]:
# What if I will add one more variable as horsepower_3. What will happen to coeeficients then?
#Please note that though the variable cofficients have changed there was no impact on accuracy (R2) of the model

# VIF

In [36]:
# The generally accepted cut-off for VIF is 2.5, with higher values denoting levels of multicollinearity
# that could negatively impact the regression model.

from statsmodels.stats.outliers_influence import variance_inflation_factor

def calc_vif(df):

    # Calculating VIF
    vif = pd.DataFrame()
    vif["variables"] = df.columns
    vif["VIF"] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return(vif)

In [37]:
#We have very high VIF value as the two variables are perfectly related to each other
calc_vif(X2).round(1).sort_values(by='VIF', ascending=False)

,variables,VIF
1,horsepower,inf
2,horsepower_2,inf
0,const,8.0


In [38]:
#Why are we not using correlation?
#What if there are many variables? There will be multiple correlation values for each variable.

In [39]:
calc_vif(X).round(1).sort_values(by='VIF', ascending=False)

,variables,VIF
0,const,8.0
1,horsepower,1.0
